# Lựa chọn đặc trưng bằng SelectKBest

Notebook này thử nhiều giá trị k để chọn ra số lượng đặc trưng hợp lý cho bài toán dự đoán tuổi Abalone.

## 1. Import thư viện

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

## 2. Nạp dữ liệu và kiểm tra nhanh

In [ ]:
# Thử nhiều đường dẫn để tránh lỗi thư mục làm việc khác nhau
duong_dan_ung_vien = [
    Path("../../data/raw/abalone.csv"),
    Path("../data/raw/abalone.csv"),
    Path("data/raw/abalone.csv"),
    Path("AbaloneAge/data/raw/abalone.csv"),
]

duong_dan_du_lieu = None
for p in duong_dan_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        duong_dan_du_lieu = p_day_du
        break

if duong_dan_du_lieu is None:
    raise FileNotFoundError(
        "Không tìm thấy file abalone.csv. Đã thử: "
        + ", ".join(str(p.resolve()) for p in duong_dan_ung_vien)
    )

df = pd.read_csv(duong_dan_du_lieu, header=None)
df.columns = [
    "sex",
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
    "rings",
]

print("Đường dẫn dữ liệu:", duong_dan_du_lieu)
print("Kích thước:", df.shape)
print("\nKiểu dữ liệu:")
print(df.dtypes)
print("\nThiếu dữ liệu mỗi cột:")
print(df.isnull().sum())
df.head()

## 3. Chia train, validation, test

In [ ]:
X = df.drop(columns=["rings"])
y = df["rings"]

# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 4. Tiền xử lý + xây quy trình chọn đặc trưng

In [ ]:
cot_so = [
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
]
cot_hang_muc = ["sex"]

bien_doi_so = Pipeline(steps=[
    ("dien_khuyet", SimpleImputer(strategy="median")),
    ("chuan_hoa", StandardScaler()),
])

bien_doi_hang_muc = Pipeline(steps=[
    ("dien_khuyet", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

tien_xu_ly = ColumnTransformer(transformers=[
    ("so", bien_doi_so, cot_so),
    ("hang_muc", bien_doi_hang_muc, cot_hang_muc),
])

print("Đã tạo xong Pipeline tiền xử lý.")

## 5. Chạy SelectKBest với nhiều giá trị k

In [ ]:
# Sau one-hot, tổng số đặc trưng sẽ tăng, nên thử k trong khoảng vừa phải
ds_k = [3, 5, 7, 9, 10]
ket_qua = []

for k in ds_k:
    mo_hinh = Pipeline(steps=[
        ("tien_xu_ly", tien_xu_ly),
        ("chon_k", SelectKBest(score_func=f_regression, k=k)),
        ("hoi_quy", Ridge(alpha=1.0)),
    ])

    mo_hinh.fit(X_train, y_train)
    du_doan_val = mo_hinh.predict(X_val)

    mae = mean_absolute_error(y_val, du_doan_val)
    rmse = np.sqrt(mean_squared_error(y_val, du_doan_val))
    r2 = r2_score(y_val, du_doan_val)

    ket_qua.append({
        "k": k,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

bang_ket_qua = pd.DataFrame(ket_qua).sort_values(by="RMSE")
bang_ket_qua

## 6. Trực quan hóa kết quả theo k

In [ ]:
bang_plot = bang_ket_qua.sort_values(by="k")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(bang_plot["k"], bang_plot["MAE"], marker="o")
axes[0].set_title("MAE theo k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("MAE")

axes[1].plot(bang_plot["k"], bang_plot["RMSE"], marker="o", color="tomato")
axes[1].set_title("RMSE theo k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("RMSE")

axes[2].plot(bang_plot["k"], bang_plot["R2"], marker="o", color="seagreen")
axes[2].set_title("R2 theo k")
axes[2].set_xlabel("k")
axes[2].set_ylabel("R2")

plt.tight_layout()
plt.savefig("../../outputs/figures/03_feature_selection_kbest_metrics.png", dpi=300, bbox_inches="tight")
plt.show()
print("Da luu hinh: 03_feature_selection_kbest_metrics.png")

## 7. Huấn luyện lại với k tốt nhất và xem đặc trưng được chọn

In [ ]:
k_tot_nhat = int(bang_ket_qua.iloc[0]["k"])
print("k tốt nhất theo RMSE:", k_tot_nhat)

mo_hinh_tot_nhat = Pipeline(steps=[
    ("tien_xu_ly", tien_xu_ly),
    ("chon_k", SelectKBest(score_func=f_regression, k=k_tot_nhat)),
    ("hoi_quy", Ridge(alpha=1.0)),
])

mo_hinh_tot_nhat.fit(X_train, y_train)

# Lấy tên đặc trưng sau tiền xử lý
ten_dac_trung_sau_xu_ly = mo_hinh_tot_nhat.named_steps["tien_xu_ly"].get_feature_names_out()
bo_loc = mo_hinh_tot_nhat.named_steps["chon_k"].get_support()
diem = mo_hinh_tot_nhat.named_steps["chon_k"].scores_

bang_diem = pd.DataFrame({
    "dac_trung": ten_dac_trung_sau_xu_ly,
    "diem_f": diem,
    "duoc_chon": bo_loc,
}).sort_values(by="diem_f", ascending=False)

bang_diem.head(15)

## 8. Đánh giá trên tập test

In [ ]:
du_doan_test = mo_hinh_tot_nhat.predict(X_test)
mae_test = mean_absolute_error(y_test, du_doan_test)
rmse_test = np.sqrt(mean_squared_error(y_test, du_doan_test))
r2_test = r2_score(y_test, du_doan_test)

print("Ket qua tren test:")
print(f"MAE : {mae_test:.4f}")
print(f"RMSE: {rmse_test:.4f}")
print(f"R2  : {r2_test:.4f}")

## 9. Lưu kết quả

In [ ]:
duong_dan_metrics = Path("../../outputs/metrics").resolve()
duong_dan_metrics.mkdir(parents=True, exist_ok=True)

bang_ket_qua.to_csv(duong_dan_metrics / "03_kbest_k_comparison.csv", index=False)
bang_diem.to_csv(duong_dan_metrics / "03_kbest_feature_scores.csv", index=False)

tom_tat = {
    "phuong_phap": "select_k_best",
    "k_tot_nhat": k_tot_nhat,
    "val_tot_nhat": bang_ket_qua.iloc[0].to_dict(),
    "test": {
        "MAE": mae_test,
        "RMSE": rmse_test,
        "R2": r2_test,
    },
}

with open(duong_dan_metrics / "03_kbest_summary.json", "w", encoding="utf-8") as f:
    json.dump(tom_tat, f, ensure_ascii=False, indent=2)

print("Da luu: 03_kbest_k_comparison.csv")
print("Da luu: 03_kbest_feature_scores.csv")
print("Da luu: 03_kbest_summary.json")